# 04A_ERA5_Extraction
Extract ERA5 weather variables from image_metadata.csv.

In [ ]:

!pip -q install earthengine-api pandas

import ee
import pandas as pd
from google.colab import files

ee.Authenticate()
ee.Initialize(project="heatwave-flood-prediction")

print("Upload image_metadata.csv")
uploaded=files.upload()
fname=list(uploaded.keys())[0]
df=pd.read_csv(fname)

rows=[]

for _,r in df.iterrows():
    pt=ee.Geometry.Point([r["longitude"],r["latitude"]])
    d=ee.Date(r["image_date"])

    img=(ee.ImageCollection("ECMWF/ERA5/DAILY")
           .filterDate(d,d.advance(1,"day"))
           .first())

    vals=img.reduceRegion(
        reducer=ee.Reducer.mean(),
        geometry=pt,
        scale=27830,
        maxPixels=1e9
    ).getInfo()

    rows.append({
        "event_id":r["event_id"],
        "image_date":r["image_date"],
        "era5_temp_mean_c":vals.get("mean_2m_air_temperature",None),
        "era5_dewpoint_mean_c":vals.get("dewpoint_2m_temperature",None),
        "era5_surface_pressure":vals.get("surface_pressure",None),
        "era5_msl_pressure":vals.get("mean_sea_level_pressure",None),
        "era5_u_wind":vals.get("u_component_of_wind_10m",None),
        "era5_v_wind":vals.get("v_component_of_wind_10m",None),
        "era5_precip":vals.get("total_precipitation",None)
    })

out=pd.DataFrame(rows)
for c in ["era5_temp_mean_c","era5_dewpoint_mean_c"]:
    out[c]=out[c]-273.15

out.to_csv("era5_features.csv",index=False)
files.download("era5_features.csv")
print(out.head())
